# WorldTrace coverage scan (run on Google Colab, not on the laptop)

**What this answers:** which of the project's cities WorldTrace covers, with how many GPS traces, and where else in the
world it is dense. It reads the dataset's metadata archive only (7.8 GB) and writes one small file, `coverage.json`,
to be saved as `evidence/worldtrace/coverage.json`. Nothing is replayed and no trace is copied out here.

**The dataset.** WorldTrace, Hugging Face `OpenTrace/WorldTrace` (revision `8fb49c023b8a2571d4cbec07681e069477743eb5`,
read 2026-10-05). Licence as stated on its card: "The original data of this dataset follows the OpenStreetMap Open
Database License (ODbL) version 1.0." Cite: Zhu, Yu, Zhao, et al., "UniTraj: Learning a Universal Trajectory Foundation
Model from Billion-Scale Worldwide Traces" (2025). The traces are public OpenStreetMap GPS uploads from August 2021 to
December 2023: **recorded, never live.**

**Privacy.** Each metadata record holds the uploader's name (`Owner`), a description and tags. This notebook does not
read, count, print or store those fields. It keeps only: a trace's number, the year it was uploaded, its point count
and whether its path enters a city's box.

In [ ]:
!pip -q install huggingface_hub
from huggingface_hub import hf_hub_download
REVISION = "8fb49c023b8a2571d4cbec07681e069477743eb5"
meta_zip = hf_hub_download("OpenTrace/WorldTrace", "Meta.zip", repo_type="dataset", revision=REVISION, local_dir="/content/worldtrace")
import os; print(meta_zip, round(os.path.getsize(meta_zip) / 1e9, 2), "GB")

The boxes below are the built areas in `config/cities/<city_id>.yaml` (south, west, north, east) on 2026-10-05.
Regenerate this cell when a city is added:
`PYTHONPATH=backend python -c "from app.data import city_config as c; print({x.city_id: list(x.bounding_box.tuple) for x in map(c.load, c.list_ids()) if x.bounding_box})"`

In [ ]:
CITY_BOXES = {
 "dhaka": [
  23.723,
  90.372,
  23.762,
  90.408
 ],
 "los_angeles": [
  34.118,
  -118.377,
  34.158,
  -118.337
 ],
 "sylhet": [
  24.87922,
  91.84852,
  24.91922,
  91.88851
 ]
}

In [ ]:
import json, re, zipfile, collections, datetime

def inside(lat, lon, box):
    s, w, n, e = box
    return s <= lat <= n and w <= lon <= e

per_city = {c: {"traces_entering_the_box": 0, "traces_starting_in_the_box": 0, "points_of_those_traces": 0, "by_upload_year": collections.Counter(), "trace_numbers": []} for c in CITY_BOXES}
grid = collections.Counter()          # 1-degree cells by start point: where in the world the dataset is dense
read = unreadable = 0
with zipfile.ZipFile(meta_zip) as z:
    for name in z.namelist():
        if not name.endswith(".json"):
            continue
        try:
            m = json.loads(z.read(name))
            lat0, lon0 = float(m["Start coordinate"][0]), float(m["Start coordinate"][1])
            path = m.get("geometry") or []                     # [longitude, latitude] pairs
        except Exception:
            unreadable += 1
            continue
        read += 1
        grid[(int(lat0 // 1), int(lon0 // 1))] += 1
        year = (re.search(r"(20\d\d)", str(m.get("Uploaded", ""))) or [None, "unknown"])[1]
        for city, box in CITY_BOXES.items():
            if inside(lat0, lon0, box) or any(inside(p[1], p[0], box) for p in path):
                c = per_city[city]
                c["traces_entering_the_box"] += 1
                c["traces_starting_in_the_box"] += int(inside(lat0, lon0, box))
                c["points_of_those_traces"] += int(m.get("Points") or 0)
                c["by_upload_year"][year] += 1
                c["trace_numbers"].append(name.rsplit("/", 1)[-1][:-5])
        if read % 200000 == 0:
            print(read, "records read")
print(read, "records read;", unreadable, "could not be read")

In [ ]:
dense = [{"lat_from": la, "lon_from": lo, "traces_starting_here": n} for (la, lo), n in grid.most_common(300)]
coverage = {
    "what": "WorldTrace coverage of the project's city boxes, counted from the dataset's metadata archive",
    "dataset": "OpenTrace/WorldTrace", "revision": REVISION, "licence": "ODbL 1.0", "time_span_stated_by_the_dataset": "August 2021 to December 2023",
    "scanned_utc": datetime.datetime.utcnow().isoformat() + "Z", "records_read": read, "records_unreadable": unreadable,
    "rule": "a trace counts for a city when its start point or any point of its path lies inside the city's box",
    "cities": {c: {**v, "by_upload_year": dict(v["by_upload_year"])} for c, v in per_city.items()},
    "one_degree_cells_with_100_or_more_traces": sum(1 for n in grid.values() if n >= 100),
    "one_degree_cells_with_any_trace": len(grid),
    "densest_one_degree_cells": dense,
    "not_done": ["cells are not turned into city names", "a trace entering a box may spend only seconds there", "uploader names, descriptions and tags were not read"],
}
json.dump(coverage, open("/content/coverage.json", "w"), indent=1)
for c, v in coverage["cities"].items():
    print(c, v["traces_entering_the_box"], "traces enter the box;", v["traces_starting_in_the_box"], "start in it;", v["by_upload_year"])
print(coverage["one_degree_cells_with_100_or_more_traces"], "one-degree cells hold 100 or more traces")

In [ ]:
from google.colab import files
files.download("/content/coverage.json")      # save it as evidence/worldtrace/coverage.json in the repository